In [0]:
%sql
CREATE TABLE IF NOT EXISTS capstone.silver.customers_scd2 (
  customer_key BIGINT,
  customer_id STRING,
  name STRING,
  email STRING,
  region STRING,
  hash_value STRING,
  start_date TIMESTAMP,
  end_date TIMESTAMP,
  is_current BOOLEAN,
  _ingest_timestamp TIMESTAMP,
  _source_file_name STRING
) USING DELTA;

In [0]:
%skip
%sql
TRUNCATE TABLE capstone.silver.customers_scd2;
INSERT INTO capstone.silver.customers_scd2
VALUES(64,'CUST78901','Jane Doe', 'jane.doe@email.com', 'West Coast', sha2(concat_ws('|', 'CUST78901','jane.doe@email.com', 'West Coast'), 256), current_timestamp(), current_timestamp(), True, current_timestamp(), 'customers.json');

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW staged_customers AS
SELECT
  -- row_number() OVER (ORDER BY customer_id) AS customer_key,
  customer_id,
  name,
  contact.email,
  region,
  sha2(concat_ws('|', name, contact.email, region), 256) AS hash_value,
  current_timestamp() AS effective_ts,
  _ingest_timestamp,
  _source_file_name
FROM capstone.bronze.customers;


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staged_customers_1 AS
WITH maxkey_cte AS (
  SELECT 
    COALESCE(MAX(customer_key), 0) AS maxkey
  FROM capstone.silver.customers_scd2
)
SELECT src.*,
CASE WHEN scd.customer_id IS NULL THEN (row_number() OVER (ORDER BY scd.customer_id)) + maxkey_cte.maxkey  ELSE scd.customer_key END AS customer_key
FROM staged_customers as src
LEFT JOIN capstone.silver.customers_scd2 as scd
ON scd.customer_id = src.customer_id 
  AND scd.is_current = TRUE
  AND scd.hash_value = src.hash_value
CROSS JOIN maxkey_cte;

In [0]:
%sql

MERGE INTO capstone.silver.customers_scd2 AS target
USING staged_customers_1 AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN MATCHED AND target.hash_value <> source.hash_value
  THEN UPDATE SET target.is_current = FALSE, target.end_date = current_timestamp();


MERGE INTO capstone.silver.customers_scd2 AS target
USING staged_customers_1 AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN NOT MATCHED
  THEN INSERT (customer_key, customer_id, name, email, region, hash_value, start_date, end_date, is_current, _ingest_timestamp, _source_file_name)
  VALUES (source.customer_key, source.customer_id, source.name, source.email, source.region, source.hash_value, current_timestamp(), NULL, TRUE, source._ingest_timestamp, source._source_file_name);

In [0]:
print("Silver Table Counts:")
print("capstone.silver.customer: ", spark.table("capstone.silver.customers_scd2").count())